# 04 · Menu Engineering & Price Elasticity
Quadrants (margin × velocity — Kasavana-Smith) exactly as `/analytics/menu-engineering`, then a **log-log elasticity** fit and profit-maximizing price per wine (Lerner rule), with bootstrap confidence intervals.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from wineops_data import get_checks, get_tables, get_consumption, get_orders, get_inventory, daily_series
plt.rcParams['figure.figsize'] = (11, 4)

inv, cons = get_inventory(), get_consumption()
vel = cons.groupby('master_wine_id')['quantity'].sum() / max((pd.to_datetime(cons['created_at'], format='ISO8601').max() - pd.to_datetime(cons['created_at'], format='ISO8601').min()).days, 1)
df = inv.set_index('master_wine_id')
df['velocity'] = vel; df = df.dropna(subset=['velocity'])
df['margin'] = df['unit_price'] - df['unit_cost']
mv, mm = df['velocity'].median(), df['margin'].median()
df['quadrant'] = np.select(
    [(df['velocity']>mv)&(df['margin']>mm), (df['velocity']>mv), (df['margin']>mm)],
    ['star','plowhorse','puzzle'], default='dog')
plt.scatter(df['velocity'], df['margin'], c=df['quadrant'].map({'star':'green','plowhorse':'orange','puzzle':'blue','dog':'red'}))
for i, r in df.iterrows(): plt.annotate(r['wine_name'][:12], (r['velocity'], r['margin']), fontsize=7)
plt.axvline(mv, ls='--', c='gray'); plt.axhline(mm, ls='--', c='gray')
plt.xlabel('bottles/day'); plt.ylabel('margin $/bottle'); plt.title('Menu engineering'); plt.show()
df[['wine_name','velocity','margin','quadrant']].sort_values('quadrant')

In [ ]:
# Elasticity needs price VARIATION. Synthetic demo: simulate weekly price tests
# (with live data, replace with wine_menu_prices history joined to consumption).
rng = np.random.default_rng(7)
rows = []
for wid, r in df.iterrows():
    base_p, base_q = r['unit_price'], max(r['velocity'], 0.05) * 7
    E_true = rng.uniform(-2.4, -0.8)   # planted elasticity to recover
    for _ in range(26):
        p = base_p * rng.uniform(0.85, 1.15)
        q = base_q * (p / base_p) ** E_true * rng.lognormal(0, 0.12)
        rows.append({'wine': r['wine_name'], 'wid': wid, 'price': p, 'qty': q, 'E_true': E_true})
px = pd.DataFrame(rows)

In [ ]:
# Log-log OLS per wine: slope IS the constant elasticity
import statsmodels.api as sm
est = []
for wid, g in px.groupby('wid'):
    model = sm.OLS(np.log(g['qty']), sm.add_constant(np.log(g['price']))).fit()
    ci = model.conf_int().loc['price']
    est.append({'wine': g['wine'].iloc[0], 'E_hat': model.params['price'],
                'lo': ci[0], 'hi': ci[1], 'E_true': g['E_true'].iloc[0]})
est = pd.DataFrame(est).sort_values('E_hat')
est.round(2)

In [ ]:
# Lerner rule: P* = MC · E/(1+E) for elastic wines (E < -1)
est2 = est.merge(df.reset_index()[['wine_name','unit_cost','unit_price']], left_on='wine', right_on='wine_name')
elastic = est2[est2['E_hat'] < -1].copy()
elastic['P_star'] = elastic['unit_cost'] * elastic['E_hat'] / (1 + elastic['E_hat'])
elastic['move_pct'] = (elastic['P_star'] / elastic['unit_price'] - 1) * 100
elastic[['wine','E_hat','unit_price','P_star','move_pct']].round(2).sort_values('move_pct')

**Read:** wines with |E| < 1 (inelastic) tolerate price increases with little volume loss — cross-reference the *plowhorse* quadrant for the highest-impact repricing list. The production endpoint `/analytics/recommendations` fires the deterministic version of exactly this rule.